In [2]:
import numpy as np

class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.pmt_positions = self.generate_pmt_positions()
        self.pmt_angular_radius = np.arcsin(self.r / self.R)

    def generate_pmt_positions(self):
        positions = []
        phi = np.pi * (np.sqrt(5.0) - 1.0)
        for i in range(self.N):
            y = 1.0 - (i / float(self.N - 1)) * 2.0
            radius_at_y = np.sqrt(1.0 - y * y)
            theta = phi * i
            x = np.cos(theta) * radius_at_y
            z = np.sin(theta) * radius_at_y
            pos = np.array([x, y, z]) * self.R
            positions.append(pos)
        return np.array(positions)

    def check_hits_vectorized(self, hit_points):
        norm_hits = hit_points / np.linalg.norm(hit_points, axis=1, keepdims=True)
        norm_pmts = self.pmt_positions / self.R
        cos_angles = np.dot(norm_hits, norm_pmts.T)
        cos_angles = np.clip(cos_angles, -1.0, 1.0)
        angles = np.arccos(cos_angles)
        
        hits_mask = angles <= self.pmt_angular_radius
        hit_indices = np.argmax(hits_mask, axis=1)
        has_hit = np.any(hits_mask, axis=1)
        hit_indices[~has_hit] = -1
        
        valid_hits = hit_indices[hit_indices != -1]
        return np.bincount(valid_hits, minlength=self.N)


class SmartSource:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_photons_generator(self, energy, pos=(0.0, 0.0, 0.0), batch_size=100):
        mean_photons = energy * self.Y
        n_photons = np.random.poisson(mean_photons)
        
        phi = np.random.uniform(0, 2 * np.pi, n_photons)
        costheta = np.random.uniform(-1.0, 1.0, n_photons)
        sintheta = np.sqrt(1.0 - costheta**2)
        
        directions = np.column_stack((sintheta * np.cos(phi), sintheta * np.sin(phi), costheta))
        positions = np.tile(pos, (n_photons, 1))
        
        for i in range(0, n_photons, batch_size):
            end_idx = min(i + batch_size, n_photons)
            yield positions[i:end_idx], directions[i:end_idx]


# с затуханием
def simulate_event_with_attenuation(detector, source, energy, source_pos=(0.0, 0.0, 0.0), attenuation_length=8.0):
    pmt_hits = np.zeros(detector.N, dtype=int)
    
    for r0, directions in source.generate_photons_generator(energy, pos=source_pos, batch_size=100):
        b = 2 * np.sum(r0 * directions, axis=1)
        c = np.sum(r0**2, axis=1) - detector.R**2
        t = (-b + np.sqrt(b**2 - 4 * c)) / 2.0
        
        hit_points = r0 + directions * t[:, np.newaxis]
        
        distances = np.linalg.norm(hit_points - r0, axis=1)
        
        survival_prob = np.exp(-distances / attenuation_length)
        survived = np.random.uniform(0, 1, len(distances)) < survival_prob
        
        if np.any(survived):
            survived_hits = hit_points[survived]
            pmt_hits += detector.check_hits_vectorized(survived_hits)
            
    pe_per_pmt = np.random.poisson(detector.a * pmt_hits)
    return np.sum(pe_per_pmt)


# без затухания
def simulate_event_no_attenuation(detector, source, energy, source_pos=(0.0, 0.0, 0.0)):
    pmt_hits = np.zeros(detector.N, dtype=int)
    for r0, directions in source.generate_photons_generator(energy, pos=source_pos, batch_size=100):
        b = 2 * np.sum(r0 * directions, axis=1)
        c = np.sum(r0**2, axis=1) - detector.R**2
        t = (-b + np.sqrt(b**2 - 4 * c)) / 2.0
        hit_points = r0 + directions * t[:, np.newaxis]
        pmt_hits += detector.check_hits_vectorized(hit_points)
        
    pe_per_pmt = np.random.poisson(detector.a * pmt_hits)
    return np.sum(pe_per_pmt)

detector = Detector(R=2.0)
source = SmartSource(Y=10000)

energy = 1.0       # 1 МэВ
n_events = 500     
z_positions = [0.0, 0.5, 1.0, 1.5]  # разные точки вдоль вертикальной оси z

print(f"Сравнение отклика для E = {energy} МэВ с затуханием и без него")
for z in z_positions:
    pos = (0.0, 0.0, z)
    
    # с затуханием
    charges_att = [simulate_event_with_attenuation(detector, source, energy, source_pos=pos, attenuation_length=8.0) for _ in range(n_events)]
    mean_att = np.mean(charges_att)
    
    # без затухания
    charges_no_att = [simulate_event_no_attenuation(detector, source, energy, source_pos=pos) for _ in range(n_events)]
    mean_no_att = np.mean(charges_no_att)
    
    loss_percent = (1.0 - mean_att / mean_no_att) * 100
    print(f"Позиция z = {z:3.1f} м | Без затухания: {mean_no_att:6.1f} ф.э. | С затуханием: {mean_att:6.1f} ф.э. | Потери: {loss_percent:.1f}%")

Сравнение отклика для E = 1.0 МэВ с затуханием и без него
Позиция z = 0.0 м | Без затухания:  124.7 ф.э. | С затуханием:   97.4 ф.э. | Потери: 21.9%
Позиция z = 0.5 м | Без затухания:  125.3 ф.э. | С затуханием:   96.3 ф.э. | Потери: 23.1%
Позиция z = 1.0 м | Без затухания:  123.7 ф.э. | С затуханием:   98.8 ф.э. | Потери: 20.2%
Позиция z = 1.5 м | Без затухания:  122.2 ф.э. | С затуханием:  101.6 ф.э. | Потери: 16.8%
